# 07 — Spectral Graph Partitioning


In [ ]:
import json
import sys
from collections import defaultdict
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import pymetis
import seaborn as sns
from matplotlib.patches import Patch
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import eigsh
from sklearn.cluster import KMeans

from build_optimiser.config import Config
from build_optimiser.graph import attach_metrics, load_graph
from build_optimiser.simulation import rebuild_cost, simulate_split

cfg = Config.from_yaml('../config.yaml')
file_df = pd.read_parquet('../data/processed/file_metrics.parquet')
target_df = pd.read_parquet('../data/processed/target_metrics.parquet')
edge_df = pd.read_parquet('../data/processed/edge_list.parquet')
G = load_graph('../data/processed/edge_list.parquet')
attach_metrics(G, target_df)

%matplotlib inline
sns.set_theme(style='whitegrid')

## Intra-Target Dependency Graph

For split candidates, build file-level include graph from header trees.


In [ ]:
# ── Split candidate selection ────────────────────────────────────────────────
EXCLUDE_TYPES = {'INTERFACE_LIBRARY', 'UTILITY'}
working_days = cfg.git_history_months * 20


def safe_int(val, default=0):
    """Convert a value to int, treating NaN/NA/None as default."""
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return default
    try:
        return int(val)
    except (ValueError, TypeError):
        return default


cost_rows = []
for _, row in target_df.iterrows():
    t = row['cmake_target']
    if row.get('target_type') in EXCLUDE_TYPES or '::' in t:
        continue
    if t not in G:
        continue
    commit_count = safe_int(row.get('git_commit_count_total'))
    change_prob = min(commit_count / working_days, 1.0) if working_days > 0 else 0.0
    r_cost = rebuild_cost(G, t, target_df)
    cost_rows.append({
        'cmake_target': t,
        'target_type': row.get('target_type'),
        'authored_file_count': safe_int(row.get('authored_file_count')),
        'codegen_file_count': safe_int(row.get('codegen_file_count')),
        'compile_time_sum_ms': safe_int(row.get('compile_time_sum_ms')),
        'total_build_time_ms': safe_int(row.get('total_build_time_ms')),
        'betweenness_centrality': float(row.get('betweenness_centrality') or 0),
        'change_prob': change_prob,
        'rebuild_cost_ms': r_cost,
        'expected_daily_cost_ms': change_prob * r_cost,
        'has_codegen': safe_int(row.get('codegen_file_count')) > 0,
    })

cost_df = pd.DataFrame(cost_rows)

# Adaptive thresholds: try strict first, relax if needed
MIN_AUTHORED_FILES = 6
MIN_COMPILE_TIME_MS = 500
split_candidates = (
    cost_df[(cost_df['authored_file_count'] >= MIN_AUTHORED_FILES)
            & (cost_df['compile_time_sum_ms'] >= MIN_COMPILE_TIME_MS)]
    .sort_values('expected_daily_cost_ms', ascending=False)
    .head(10)
    .copy()
)

if split_candidates.empty:
    # Relax: any target with >= 2 authored files and some compile time
    MIN_AUTHORED_FILES = 2
    MIN_COMPILE_TIME_MS = 0
    split_candidates = (
        cost_df[(cost_df['authored_file_count'] >= MIN_AUTHORED_FILES)
                & (cost_df['compile_time_sum_ms'] > MIN_COMPILE_TIME_MS)]
        .sort_values('expected_daily_cost_ms', ascending=False)
        .head(10)
        .copy()
    )
    print(f"Relaxed thresholds to authored_files >= {MIN_AUTHORED_FILES}, compile_time > 0")

print(f"Split candidates: {len(split_candidates)} "
      f"(from {len(cost_df)} analysable targets)")
display(split_candidates[['cmake_target', 'authored_file_count', 'codegen_file_count',
                           'compile_time_sum_ms', 'betweenness_centrality',
                           'expected_daily_cost_ms', 'has_codegen']].reset_index(drop=True))


# ── Intra-target file graph builder ─────────────────────────────────────────
def parse_header_tree(json_str) -> list[tuple[int, str]]:
    """Decode the JSON header_tree column to a list of (depth, path) pairs."""
    if json_str is None or (isinstance(json_str, float) and pd.isna(json_str)):
        return []
    try:
        if pd.isna(json_str):
            return []
    except (ValueError, TypeError):
        pass
    try:
        return [(int(d), str(p)) for d, p in json.loads(json_str)]
    except (json.JSONDecodeError, ValueError, TypeError):
        return []


def build_intra_target_graph(target_name: str, file_df: pd.DataFrame) -> nx.DiGraph:
    """Build a file-level include graph for one target.

    Nodes = source files belonging to this target.
    Edges = A -> B if A includes B (matching the project's "A depends on B" convention).
    Only intra-target edges are kept; cross-target headers are ignored.
    """
    target_files = file_df[file_df['cmake_target'] == target_name].copy()
    if target_files.empty:
        return nx.DiGraph()

    source_paths = set(target_files['source_file'].dropna())
    G_file = nx.DiGraph()

    # Add nodes with attributes
    for _, row in target_files.iterrows():
        sf = row['source_file']
        if pd.isna(sf):
            continue
        G_file.add_node(sf,
                        is_generated=bool(row.get('is_generated', False)),
                        compile_time_ms=safe_int(row.get('compile_time_ms')),
                        code_lines=safe_int(row.get('code_lines')),
                        git_churn=safe_int(row.get('git_churn')))

    # Parse header trees and reconstruct include edges via stack-based parent tracking
    for _, row in target_files.iterrows():
        source = row['source_file']
        if pd.isna(source):
            continue
        tree = parse_header_tree(row.get('header_tree'))
        if not tree:
            continue
        # Stack: (depth, path). Depth 0 = the compile unit itself.
        stack: list[tuple[int, str]] = [(0, source)]
        for depth, header_path in tree:
            while len(stack) > 1 and stack[-1][0] >= depth:
                stack.pop()
            parent_path = stack[-1][1]
            # Only keep intra-target edges
            if parent_path in source_paths and header_path in source_paths:
                if not G_file.has_edge(parent_path, header_path):
                    G_file.add_edge(parent_path, header_path, weight=0.0)
            stack.append((depth, header_path))

    # Co-change edge weighting: proxy using git_churn
    churns = nx.get_node_attributes(G_file, 'git_churn')
    max_churn = max(churns.values()) if churns else 1
    if max_churn == 0:
        max_churn = 1
    for u, v in G_file.edges():
        cu, cv = churns.get(u, 0), churns.get(v, 0)
        G_file[u][v]['weight'] = min(cu, cv) / max_churn

    return G_file


# ── Build graphs for all candidates and show primary ────────────────────────
G_file_map: dict[str, nx.DiGraph] = {}
for _, row in split_candidates.iterrows():
    t = row['cmake_target']
    G_file_map[t] = build_intra_target_graph(t, file_df)

primary = split_candidates.iloc[0]['cmake_target']
gf = G_file_map[primary]
n_components = nx.number_weakly_connected_components(gf)
print(f"\nPrimary candidate: {primary}")
print(f"  Nodes: {gf.number_of_nodes()}, Edges: {gf.number_of_edges()}")
print(f"  Weakly connected components: {n_components}")
print(f"  Isolated nodes: {sum(1 for n in gf.nodes() if gf.degree(n) == 0)}")
gen_count = sum(1 for n in gf.nodes() if gf.nodes[n].get('is_generated', False))
print(f"  Generated files: {gen_count}")

# ── Visualisation: file-level graph for primary candidate ────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

ax = axes[0]
if gf.number_of_nodes() > 0:
    k_val = 2.0 / np.sqrt(max(gf.number_of_nodes(), 1))
    pos = nx.spring_layout(gf, k=k_val, iterations=30, seed=42)
    node_colors = ['darkorange' if gf.nodes[n].get('is_generated') else 'steelblue'
                   for n in gf.nodes()]
    ct_vals = [gf.nodes[n].get('compile_time_ms', 1) for n in gf.nodes()]
    max_ct = max(ct_vals) if ct_vals else 1
    node_sizes = [30 + 200 * (ct / max(max_ct, 1)) for ct in ct_vals]
    nx.draw_networkx(gf, pos=pos, ax=ax, node_color=node_colors, node_size=node_sizes,
                     edge_color='grey', alpha=0.7, arrows=True, arrowsize=8,
                     with_labels=gf.number_of_nodes() <= 30,
                     font_size=6, width=0.5,
                     labels={n: Path(n).name for n in gf.nodes()} if gf.number_of_nodes() <= 30 else {})
    ax.legend(handles=[Patch(color='steelblue', label='Authored'),
                       Patch(color='darkorange', label='Generated')], loc='upper left')
ax.set_title(f'Intra-Target Include Graph: {primary}\n'
             f'({gf.number_of_nodes()} files, {gf.number_of_edges()} include edges)')
ax.axis('off')

# Right panel: candidate overview bar chart
ax = axes[1]
sc = split_candidates.sort_values('expected_daily_cost_ms', ascending=True)
colors = ['darkorange' if c else 'steelblue' for c in sc['has_codegen']]
ax.barh(range(len(sc)), sc['expected_daily_cost_ms'].values, color=colors)
ax.set_yticks(range(len(sc)))
ax.set_yticklabels(sc['cmake_target'].values, fontsize=8)
ax.set_xlabel('Expected daily rebuild cost (ms)')
ax.set_title('Split Candidates by Expected Rebuild Cost')
ax.legend(handles=[Patch(color='steelblue', label='Authored-only'),
                   Patch(color='darkorange', label='Has codegen')], loc='lower right')

plt.tight_layout()
plt.show()

## Spectral Partitioning

Fiedler vector for 2-way partition. k eigenvectors + k-means for k-way.


In [ ]:
# ── Constraint handling: .cpp/.h pair contraction ────────────────────────────
def contract_cpp_header_pairs(G_file: nx.DiGraph) -> tuple[nx.Graph, dict[str, str]]:
    """Contract .cpp/.h pairs sharing a stem into single super-nodes.

    Returns an undirected contracted graph (partitioning works on undirected)
    and a mapping from original file path to super-node representative.
    """
    nodes = list(G_file.nodes())
    node_to_supernode: dict[str, str] = {}

    # Group files by (directory, stem)
    stem_map: dict[str, list[str]] = defaultdict(list)
    for path in nodes:
        p = Path(path)
        stem_map[str(p.parent / p.stem)].append(path)

    supernode_members: dict[str, list[str]] = {}
    for stem, paths in stem_map.items():
        cpp_exts = {'.cpp', '.cc', '.cxx', '.c'}
        hdr_exts = {'.h', '.hpp', '.hh', '.hxx'}
        cpp_files = [p for p in paths if Path(p).suffix in cpp_exts]
        hdr_files = [p for p in paths if Path(p).suffix in hdr_exts]
        if cpp_files and hdr_files:
            rep = cpp_files[0]
            members = cpp_files + hdr_files
            for m in members:
                node_to_supernode[m] = rep
            supernode_members[rep] = members
        else:
            for p in paths:
                node_to_supernode[p] = p
                supernode_members[p] = [p]

    # Un-paired nodes that weren't in any stem group
    for path in nodes:
        if path not in node_to_supernode:
            node_to_supernode[path] = path
            supernode_members[path] = [path]

    # Build contracted undirected graph
    G_contracted = nx.Graph()
    for sn, members in supernode_members.items():
        G_contracted.add_node(sn,
                              compile_time_ms=sum(
                                  G_file.nodes[m].get('compile_time_ms', 0)
                                  for m in members if m in G_file),
                              code_lines=sum(
                                  G_file.nodes[m].get('code_lines', 0)
                                  for m in members if m in G_file),
                              git_churn=max(
                                  (G_file.nodes[m].get('git_churn', 0)
                                   for m in members if m in G_file), default=0),
                              is_generated=any(
                                  G_file.nodes[m].get('is_generated', False)
                                  for m in members if m in G_file),
                              members=members)

    for u, v, data in G_file.edges(data=True):
        sn_u, sn_v = node_to_supernode[u], node_to_supernode[v]
        if sn_u == sn_v:
            continue
        if G_contracted.has_edge(sn_u, sn_v):
            G_contracted[sn_u][sn_v]['weight'] += data.get('weight', 0.0)
        else:
            G_contracted.add_edge(sn_u, sn_v, weight=data.get('weight', 0.0))

    return G_contracted, node_to_supernode


# ── Constraint handling: codegen co-location via heavy edges ─────────────────
def pin_generated_files(G_contracted: nx.Graph, heavy_weight: float = 1e4) -> list[str]:
    """Add heavy edges between generated super-nodes and their consumers."""
    gen_nodes = [n for n in G_contracted.nodes()
                 if G_contracted.nodes[n].get('is_generated', False)]
    for gen_node in gen_nodes:
        for nbr in list(G_contracted.neighbors(gen_node)):
            G_contracted[gen_node][nbr]['weight'] += heavy_weight
    return gen_nodes


# ── Spectral 2-way partition (Fiedler vector) ───────────────────────────────
def spectral_2way(G_contracted: nx.Graph) -> tuple[np.ndarray, dict[str, int], float]:
    """Partition at sign of the Fiedler vector (2nd smallest eigenvector of normalised Laplacian)."""
    nodes = list(G_contracted.nodes())
    n = len(nodes)
    if n < 2:
        return np.zeros(max(n, 1)), {nd: 0 for nd in nodes}, 0.0

    # Work on largest connected component if disconnected
    components = list(nx.connected_components(G_contracted))
    if len(components) > 1:
        lcc = max(components, key=len)
        G_work = G_contracted.subgraph(lcc).copy()
    else:
        G_work = G_contracted
        lcc = set(nodes)

    work_nodes = list(G_work.nodes())
    nw = len(work_nodes)
    if nw < 2:
        return np.zeros(n), {nd: 0 for nd in nodes}, 0.0

    node_idx = {nd: i for i, nd in enumerate(work_nodes)}

    # Build weighted adjacency matrix
    rows_i, cols_j, vals = [], [], []
    for u, v, data in G_work.edges(data=True):
        w = min(data.get('weight', 1.0), 100.0)  # clip heavy constraint edges for numerics
        i, j = node_idx[u], node_idx[v]
        rows_i += [i, j]
        cols_j += [j, i]
        vals += [w, w]

    if not rows_i:
        return np.zeros(n), {nd: (i % 2) for i, nd in enumerate(nodes)}, 0.0

    A = csr_matrix((vals, (rows_i, cols_j)), shape=(nw, nw))
    degree = np.array(A.sum(axis=1)).flatten()
    degree = np.where(degree == 0, 1e-10, degree)
    D_inv_sqrt = csr_matrix((1.0 / np.sqrt(degree), (range(nw), range(nw))), shape=(nw, nw))
    D = csr_matrix((degree, (range(nw), range(nw))), shape=(nw, nw))
    L_norm = D_inv_sqrt @ (D - A) @ D_inv_sqrt

    eigenvalues, eigenvectors = eigsh(L_norm, k=min(2, nw - 1), which='SM', tol=1e-6, maxiter=1000)
    idx_sorted = np.argsort(eigenvalues)
    fiedler_idx = idx_sorted[1] if len(idx_sorted) > 1 else idx_sorted[0]
    fiedler_vec = eigenvectors[:, fiedler_idx]
    fiedler_val = float(eigenvalues[fiedler_idx])

    partition = {work_nodes[i]: (0 if fiedler_vec[i] < 0 else 1) for i in range(nw)}
    # Assign nodes outside LCC to partition 0
    for nd in nodes:
        if nd not in partition:
            partition[nd] = 0

    # Full-length Fiedler vector (0.0 for non-LCC nodes)
    full_fiedler = np.zeros(n)
    for i, nd in enumerate(nodes):
        if nd in node_idx:
            full_fiedler[i] = fiedler_vec[node_idx[nd]]

    return full_fiedler, partition, fiedler_val


# ── Spectral k-way partition ────────────────────────────────────────────────
def spectral_kway(G_contracted: nx.Graph, k: int) -> dict[str, int]:
    """k-way partition via k smallest non-trivial Laplacian eigenvectors + k-means."""
    nodes = list(G_contracted.nodes())
    n = len(nodes)
    if n <= k:
        return {nd: i % k for i, nd in enumerate(nodes)}

    components = list(nx.connected_components(G_contracted))
    if len(components) > 1:
        lcc = max(components, key=len)
        G_work = G_contracted.subgraph(lcc).copy()
    else:
        G_work = G_contracted
        lcc = set(nodes)

    work_nodes = list(G_work.nodes())
    nw = len(work_nodes)
    if nw <= k:
        part = {nd: i % k for i, nd in enumerate(work_nodes)}
        for nd in nodes:
            if nd not in part:
                part[nd] = 0
        return part

    node_idx = {nd: i for i, nd in enumerate(work_nodes)}
    rows_i, cols_j, vals = [], [], []
    for u, v, data in G_work.edges(data=True):
        w = min(data.get('weight', 1.0), 100.0)
        i, j = node_idx[u], node_idx[v]
        rows_i += [i, j]
        cols_j += [j, i]
        vals += [w, w]

    if not rows_i:
        return {nd: i % k for i, nd in enumerate(nodes)}

    A = csr_matrix((vals, (rows_i, cols_j)), shape=(nw, nw))
    degree = np.array(A.sum(axis=1)).flatten()
    degree = np.where(degree == 0, 1e-10, degree)
    D_inv_sqrt = csr_matrix((1.0 / np.sqrt(degree), (range(nw), range(nw))), shape=(nw, nw))
    D = csr_matrix((degree, (range(nw), range(nw))), shape=(nw, nw))
    L_norm = D_inv_sqrt @ (D - A) @ D_inv_sqrt

    n_eig = min(k + 1, nw - 1)
    eigenvalues, eigenvectors = eigsh(L_norm, k=n_eig, which='SM', tol=1e-6, maxiter=2000)
    idx_sorted = np.argsort(eigenvalues)
    # Skip trivial eigenvector 0, use 1..k
    use_idx = idx_sorted[1:k + 1] if len(idx_sorted) > k else idx_sorted[1:]
    embedding = eigenvectors[:, use_idx]

    norms = np.linalg.norm(embedding, axis=1, keepdims=True)
    norms = np.where(norms < 1e-10, 1.0, norms)
    embedding = embedding / norms

    actual_k = min(k, embedding.shape[1] + 1, nw)
    km = KMeans(n_clusters=actual_k, random_state=42, n_init=10)
    labels = km.fit_predict(embedding)

    partition = {work_nodes[i]: int(labels[i]) for i in range(nw)}
    for nd in nodes:
        if nd not in partition:
            partition[nd] = 0
    return partition


# ── Helper: normalised cut metric ───────────────────────────────────────────
def normalised_cut(G_contracted: nx.Graph, partition: dict[str, int]) -> float:
    """Normalised cut: sum over partitions of (cut edges / total weighted degree in partition)."""
    parts = set(partition.values())
    if len(parts) <= 1:
        return 0.0
    total = 0.0
    for pid in parts:
        nodes_in = {n for n, p in partition.items() if p == pid}
        cut_w, vol = 0.0, 0.0
        for n in nodes_in:
            if n not in G_contracted:
                continue
            for nbr in G_contracted.neighbors(n):
                w = G_contracted[n][nbr].get('weight', 1.0)
                vol += w
                if partition.get(nbr, -1) != pid:
                    cut_w += w
        if vol > 0:
            total += cut_w / vol
    return total


# ── Run spectral on primary candidate ───────────────────────────────────────
G_contracted_map: dict[str, nx.Graph] = {}
supernode_map: dict[str, dict[str, str]] = {}
spectral_2way_map: dict[str, dict[str, int]] = {}
spectral_kway_map: dict[str, tuple[int, dict[str, int]]] = {}

gf = G_file_map[primary]
G_c, n2s = contract_cpp_header_pairs(gf)
gen_pinned = pin_generated_files(G_c)
G_contracted_map[primary] = G_c
supernode_map[primary] = n2s

n_contracted = G_c.number_of_nodes()
n_pairs = gf.number_of_nodes() - n_contracted
print(f"Contracted graph: {n_contracted} super-nodes ({n_pairs} pairs merged)")
print(f"Generated super-nodes with heavy edges: {len(gen_pinned)}")

# 2-way Fiedler
fiedler_vec, part_2way, fiedler_val = spectral_2way(G_c)
spectral_2way_map[primary] = part_2way
sizes = [sum(1 for v in part_2way.values() if v == p) for p in sorted(set(part_2way.values()))]
ct_per_part = []
for pid in sorted(set(part_2way.values())):
    ct = sum(G_c.nodes[n].get('compile_time_ms', 0) for n, p in part_2way.items() if p == pid and n in G_c)
    ct_per_part.append(ct)
balance = min(ct_per_part) / max(ct_per_part) if max(ct_per_part) > 0 else 0.0
cut_edges = sum(1 for u, v in G_c.edges() if part_2way.get(u, -1) != part_2way.get(v, -1))

print("\n2-way spectral partition:")
print(f"  Algebraic connectivity (Fiedler value): {fiedler_val:.6f}")
print(f"  Partition sizes: {sizes}")
print(f"  Compile time per partition: {ct_per_part} ms")
print(f"  Balance ratio (min/max): {balance:.3f}")
print(f"  Cut edges: {cut_edges}")

# k-way sweep
max_k = min(6, max(n_contracted // 3, 2))
kway_results = []
for k in range(2, max_k + 1):
    part_k = spectral_kway(G_c, k)
    ncut = normalised_cut(G_c, part_k)
    kway_results.append({'k': k, 'normalised_cut': ncut, 'partition': part_k})

# Elbow detection: pick k where marginal cut reduction drops below 10% of total
best_k = 2
if len(kway_results) >= 2:
    cuts = [r['normalised_cut'] for r in kway_results]
    total_reduction = cuts[0] - cuts[-1] if cuts[0] > cuts[-1] else 1.0
    for i in range(1, len(cuts)):
        marginal = cuts[i - 1] - cuts[i]
        if marginal < 0.1 * total_reduction:
            best_k = kway_results[i - 1]['k']
            break
    else:
        best_k = kway_results[-1]['k']

best_kway = next(r for r in kway_results if r['k'] == best_k)
spectral_kway_map[primary] = (best_k, best_kway['partition'])
print(f"\nk-way sweep (k=2..{max_k}): best k={best_k} (normalised cut={best_kway['normalised_cut']:.4f})")

# ── Visualisation: 3 panels ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Panel 1: Fiedler vector histogram
ax = axes[0]
nodes_list = list(G_c.nodes())
fv_vals = [fiedler_vec[i] for i in range(len(nodes_list))]
colors_hist = ['darkorange' if v >= 0 else 'steelblue' for v in fv_vals]
ax.bar(range(len(fv_vals)), sorted(fv_vals),
       color=[c for _, c in sorted(zip(fv_vals, colors_hist))], alpha=0.8)
ax.axhline(0, color='red', linestyle='--', linewidth=1.5, label='Cut threshold')
ax.set_xlabel('Super-node (sorted by Fiedler value)')
ax.set_ylabel('Fiedler vector component')
ax.set_title(f'Fiedler Vector — {primary}')
ax.legend()

# Panel 2: Contracted graph coloured by 2-way partition
ax = axes[1]
if G_c.number_of_nodes() > 0:
    k_val = 2.0 / np.sqrt(max(G_c.number_of_nodes(), 1))
    pos_c = nx.spring_layout(G_c, k=k_val, iterations=30, seed=42)
    node_colors_2way = ['darkorange' if part_2way.get(n, 0) == 1 else 'steelblue'
                        for n in G_c.nodes()]
    edge_colors = ['red' if part_2way.get(u, -1) != part_2way.get(v, -1) else 'lightgrey'
                   for u, v in G_c.edges()]
    ct_vals = [G_c.nodes[n].get('compile_time_ms', 1) for n in G_c.nodes()]
    max_ct = max(ct_vals) if ct_vals else 1
    node_sizes = [40 + 200 * (ct / max(max_ct, 1)) for ct in ct_vals]
    nx.draw_networkx(G_c, pos=pos_c, ax=ax, node_color=node_colors_2way,
                     node_size=node_sizes, edge_color=edge_colors, width=1.5,
                     with_labels=G_c.number_of_nodes() <= 20,
                     font_size=6,
                     labels={n: Path(n).name for n in G_c.nodes()} if G_c.number_of_nodes() <= 20 else {})
    ax.legend(handles=[Patch(color='steelblue', label='Partition 0'),
                       Patch(color='darkorange', label='Partition 1')], loc='upper left')
ax.set_title('2-way Spectral Partition (cut edges in red)')
ax.axis('off')

# Panel 3: Normalised cut vs k
ax = axes[2]
ks = [r['k'] for r in kway_results]
ncuts = [r['normalised_cut'] for r in kway_results]
ax.plot(ks, ncuts, 'o-', color='steelblue', linewidth=2, markersize=8)
ax.axvline(best_k, color='red', linestyle='--', alpha=0.7, label=f'Best k={best_k}')
ax.set_xlabel('k (number of partitions)')
ax.set_ylabel('Normalised cut')
ax.set_title('Normalised Cut vs k')
ax.set_xticks(ks)
ax.legend()

plt.tight_layout()
plt.show()

## METIS Partitioning

pymetis as alternative. Weight nodes by compile time or SLOC.


In [ ]:
# ── METIS partition via pymetis ───────────────────────────────────────────────
def metis_partition(G_contracted: nx.Graph, nparts: int,
                    node_weight_attr: str = 'compile_time_ms') -> tuple[int, dict[str, int]]:
    """Partition using METIS. Node weights from node_weight_attr, edge weights from 'weight'."""
    nodes = list(G_contracted.nodes())
    n = len(nodes)
    if n < nparts:
        return 0, {nd: i % nparts for i, nd in enumerate(nodes)}

    node_idx = {nd: i for i, nd in enumerate(nodes)}

    # Build CSR adjacency (xadj/adjncy) + eweights
    adjacency: list[list[int]] = [[] for _ in range(n)]
    edge_weights: list[list[int]] = [[] for _ in range(n)]
    for u, v, data in G_contracted.edges(data=True):
        i, j = node_idx[u], node_idx[v]
        raw_w = data.get('weight', 1.0)
        int_w = max(1, min(int(raw_w * 1000) + 1, 10000))
        adjacency[i].append(j)
        edge_weights[i].append(int_w)
        adjacency[j].append(i)
        edge_weights[j].append(int_w)

    xadj = [0]
    for row in adjacency:
        xadj.append(xadj[-1] + len(row))
    adjncy = [nb for row in adjacency for nb in row]
    flat_eweights = [w for row in edge_weights for w in row]

    # Node weights: scale down and clip to int >= 1
    vweights = []
    for nd in nodes:
        val = G_contracted.nodes[nd].get(node_weight_attr, 0) or 0
        vweights.append(max(1, int(val // 10)))

    edgecut, parts = pymetis.part_graph(
        nparts,
        xadj=xadj,
        adjncy=adjncy,
        vweights=vweights,
        eweights=flat_eweights if flat_eweights else None,
    )
    return edgecut, {nodes[i]: int(parts[i]) for i in range(n)}


# ── Helper: partition agreement (accounts for label permutation) ─────────────
def partition_agreement(part_a: dict[str, int], part_b: dict[str, int]) -> float:
    """Fraction of nodes assigned to the same group, accounting for label permutation."""
    from itertools import permutations

    common = set(part_a.keys()) & set(part_b.keys())
    if not common:
        return 0.0
    labels_b = [part_b[n] for n in common]
    unique_b = sorted(set(labels_b))
    # Try all permutations for small label sets, greedy match for larger
    if len(unique_b) <= 6:
        best = 0
        for perm in permutations(unique_b):
            mapping = dict(zip(unique_b, perm))
            matches = sum(1 for n in common if part_a[n] == mapping.get(part_b[n], -1))
            best = max(best, matches)
        return best / len(common)
    else:
        return sum(1 for n in common if part_a[n] == part_b[n]) / len(common)


# ── Run METIS on primary candidate ──────────────────────────────────────────
metis_2way_map: dict[str, dict[str, int]] = {}
metis_kway_map: dict[str, tuple[int, dict[str, int]]] = {}

G_c = G_contracted_map[primary]

# METIS 2-way with compile time weighting
edgecut_ct, metis_part_ct = metis_partition(G_c, 2, 'compile_time_ms')
metis_2way_map[primary] = metis_part_ct

# METIS 2-way with SLOC weighting
edgecut_sl, metis_part_sl = metis_partition(G_c, 2, 'code_lines')

# METIS k-way (use same best_k from spectral)
best_k_spec = spectral_kway_map[primary][0]
edgecut_k, metis_part_k = metis_partition(G_c, best_k_spec, 'compile_time_ms')
metis_kway_map[primary] = (best_k_spec, metis_part_k)

# Compare with spectral
agreement_2way = partition_agreement(spectral_2way_map[primary], metis_part_ct)
spectral_cut = sum(1 for u, v in G_c.edges()
                   if spectral_2way_map[primary].get(u) != spectral_2way_map[primary].get(v))

print("METIS 2-way (compile_time weighted):")
print(f"  Edge cut: {edgecut_ct} (spectral: {spectral_cut})")
ct_metis = []
for pid in sorted(set(metis_part_ct.values())):
    ct = sum(G_c.nodes[n].get('compile_time_ms', 0) for n, p in metis_part_ct.items() if p == pid and n in G_c)
    ct_metis.append(ct)
balance_metis = min(ct_metis) / max(ct_metis) if max(ct_metis) > 0 else 0.0
print(f"  Compile time per partition: {ct_metis} ms")
print(f"  Balance: {balance_metis:.3f}")
print(f"  Agreement with spectral 2-way: {agreement_2way:.1%}")

print("\nMETIS 2-way (SLOC weighted):")
print(f"  Edge cut: {edgecut_sl}")
sl_metis = []
for pid in sorted(set(metis_part_sl.values())):
    sl = sum(G_c.nodes[n].get('code_lines', 0) for n, p in metis_part_sl.items() if p == pid and n in G_c)
    sl_metis.append(sl)
sl_balance = min(sl_metis) / max(sl_metis) if max(sl_metis) > 0 else 0.0
print(f"  SLOC per partition: {sl_metis}")
print(f"  SLOC balance: {sl_balance:.3f}")

print(f"\nMETIS {best_k_spec}-way (compile_time weighted):")
print(f"  Edge cut: {edgecut_k}")

# ── Visualisation: 2 panels ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel 1: Side-by-side spectral vs METIS (using same layout)
ax = axes[0]
if G_c.number_of_nodes() > 0:
    if 'pos_c' not in dir():
        k_val = 2.0 / np.sqrt(max(G_c.number_of_nodes(), 1))
        pos_c = nx.spring_layout(G_c, k=k_val, iterations=30, seed=42)

    # Colour: steelblue=agree on 0, darkorange=agree on 1, green=disagree
    node_colors_cmp = []
    for n in G_c.nodes():
        sp = spectral_2way_map[primary].get(n, 0)
        mp = metis_part_ct.get(n, 0)
        if sp == mp:
            node_colors_cmp.append('steelblue' if sp == 0 else 'darkorange')
        else:
            node_colors_cmp.append('#55A868')

    nx.draw_networkx(G_c, pos=pos_c, ax=ax, node_color=node_colors_cmp,
                     node_size=60, edge_color='lightgrey', width=0.5,
                     with_labels=G_c.number_of_nodes() <= 20, font_size=6,
                     labels={n: Path(n).name for n in G_c.nodes()} if G_c.number_of_nodes() <= 20 else {})
    n_diff = sum(1 for n in G_c.nodes()
                 if spectral_2way_map[primary].get(n) != metis_part_ct.get(n))
    ax.legend(handles=[Patch(color='steelblue', label='Both: Part 0'),
                       Patch(color='darkorange', label='Both: Part 1'),
                       Patch(color='#55A868', label=f'Disagree ({n_diff})')],
              loc='upper left')
ax.set_title('Spectral vs METIS 2-way Agreement')
ax.axis('off')

# Panel 2: Grouped bars — compile time balance comparison
ax = axes[1]
methods = ['Spectral 2-way', 'METIS (compile)', 'METIS (SLOC)']
# Compute compile time per partition for each method
all_parts = [spectral_2way_map[primary], metis_part_ct, metis_part_sl]
balances = []
for part in all_parts:
    cts = []
    for pid in sorted(set(part.values())):
        ct = sum(G_c.nodes[n].get('compile_time_ms', 0) for n, p in part.items() if p == pid and n in G_c)
        cts.append(ct)
    balances.append(min(cts) / max(cts) if max(cts) > 0 else 0.0)

colors_bar = ['steelblue', 'darkorange', '#55A868']
ax.bar(methods, balances, color=colors_bar, alpha=0.8)
ax.axhline(0.5, color='red', linestyle='--', alpha=0.6, label='Target balance (0.5)')
ax.set_ylabel('Balance ratio (min/max compile time)')
ax.set_title('Compile Time Balance: Spectral vs METIS')
ax.set_ylim(0, 1.05)
ax.legend()

plt.tight_layout()
plt.show()

## Constraint Handling

.cpp/.h pairs contracted. Generated files co-located with consumers.


In [ ]:
# ── Verify codegen co-location constraint ────────────────────────────────────
# Process all candidates: build contracted graphs, run all 4 partition methods,
# and verify that generated files stay with their consumers.

for _, row in split_candidates.iterrows():
    t = row['cmake_target']
    if t == primary:
        continue  # already processed
    gf = G_file_map[t]
    if gf.number_of_nodes() < 2:
        continue
    G_c, n2s = contract_cpp_header_pairs(gf)
    pin_generated_files(G_c)
    G_contracted_map[t] = G_c
    supernode_map[t] = n2s

    # Spectral 2-way
    _, part_2, _ = spectral_2way(G_c)
    spectral_2way_map[t] = part_2

    # Spectral k-way
    max_k_t = min(6, max(G_c.number_of_nodes() // 3, 2))
    best_k_t, best_ncut = 2, float('inf')
    best_part_k = part_2
    for k in range(2, max_k_t + 1):
        pk = spectral_kway(G_c, k)
        nc = normalised_cut(G_c, pk)
        if nc < best_ncut:
            best_ncut = nc
            best_k_t = k
            best_part_k = pk
    spectral_kway_map[t] = (best_k_t, best_part_k)

    # METIS 2-way and k-way
    _, mp2 = metis_partition(G_c, 2, 'compile_time_ms')
    metis_2way_map[t] = mp2
    _, mpk = metis_partition(G_c, best_k_t, 'compile_time_ms')
    metis_kway_map[t] = (best_k_t, mpk)

# ── Codegen constraint check across all candidates and methods ───────────────
def check_codegen_constraint(target_name, partition_map, n2s_map, G_file):
    """Check if generated files are co-located with all their consumers."""
    violations = []
    gen_nodes = [n for n in G_file.nodes() if G_file.nodes[n].get('is_generated', False)]
    for gf_node in gen_nodes:
        sn = n2s_map.get(gf_node, gf_node)
        gf_part = partition_map.get(sn, -1)
        # Consumers: nodes that include this generated file (predecessors in the directed graph)
        consumers = list(G_file.predecessors(gf_node))
        consumer_parts = set()
        for c in consumers:
            c_sn = n2s_map.get(c, c)
            consumer_parts.add(partition_map.get(c_sn, -1))
        if consumer_parts and (len(consumer_parts) > 1 or gf_part not in consumer_parts):
            violations.append({
                'generated_file': Path(gf_node).name,
                'gen_partition': gf_part,
                'consumer_partitions': sorted(consumer_parts),
                'n_consumers': len(consumers),
            })
    return violations


constraint_rows = []
method_names = ['spectral_2way', 'spectral_kway', 'metis_2way', 'metis_kway']

for _, row in split_candidates.iterrows():
    t = row['cmake_target']
    if t not in G_contracted_map:
        continue
    n2s = supernode_map[t]
    gf = G_file_map[t]
    gen_count = sum(1 for n in gf.nodes() if gf.nodes[n].get('is_generated', False))

    partitions = {
        'spectral_2way': spectral_2way_map.get(t, {}),
        'spectral_kway': spectral_kway_map.get(t, (2, {}))[1],
        'metis_2way': metis_2way_map.get(t, {}),
        'metis_kway': metis_kway_map.get(t, (2, {}))[1],
    }

    for method, part in partitions.items():
        if not part:
            continue
        violations = check_codegen_constraint(t, part, n2s, gf)
        constraint_rows.append({
            'cmake_target': t,
            'method': method,
            'generated_files': gen_count,
            'violations': len(violations),
            'ok': len(violations) == 0,
            'violation_details': violations,
        })

constraint_df = pd.DataFrame(constraint_rows)

if not constraint_df.empty:
    summary = constraint_df.groupby('method').agg(
        targets_checked=('cmake_target', 'nunique'),
        all_ok=('ok', 'all'),
        total_violations=('violations', 'sum'),
    ).reset_index()
    print("Codegen co-location constraint summary by method:")
    display(summary)

    violated = constraint_df[~constraint_df['ok']]
    if not violated.empty:
        print(f"\nConstraint violations ({len(violated)} target/method combos):")
        for _, vr in violated.iterrows():
            print(f"  {vr['cmake_target']} [{vr['method']}]: "
                  f"{vr['violations']} generated file(s) separated from consumers")
            for det in vr['violation_details']:
                print(f"    {det['generated_file']}: gen in part {det['gen_partition']}, "
                      f"consumers in parts {det['consumer_partitions']}")
    else:
        print("\nAll partitions satisfy the codegen co-location constraint.")
else:
    print("No candidates with generated files to check.")

## Evaluation

Cross-partition includes, compile time balance, critical path impact.


In [ ]:
# ── Evaluation: cross-partition includes, balance, compile time saving ────────
def evaluate_partition(target_name, partition_map, n2s_map, G_file, file_df, target_df, G):
    """Evaluate a partition: cross-partition includes, compile time balance, saving."""
    # Map original files to partitions via super-nodes
    file_to_partition = {}
    for orig, sn in n2s_map.items():
        if sn in partition_map:
            file_to_partition[orig] = partition_map[sn]

    # 1. Cross-partition includes
    cross_includes = sum(
        1 for u, v in G_file.edges()
        if file_to_partition.get(u, -1) != file_to_partition.get(v, -1)
        and file_to_partition.get(u, -1) >= 0
        and file_to_partition.get(v, -1) >= 0
    )

    # 2. Compile time per partition (authored files only)
    target_files = file_df[
        (file_df['cmake_target'] == target_name) & (~file_df['is_generated'])
    ]
    partition_ids = sorted(set(file_to_partition.values()))
    compile_per_partition = []
    file_groups = []
    for pid in partition_ids:
        files_in_part = [f for f, p in file_to_partition.items()
                         if p == pid and f in set(target_files['source_file'])]
        ct_sum = int(target_files[target_files['source_file'].isin(files_in_part)]
                     ['compile_time_ms'].sum() or 0)
        compile_per_partition.append(ct_sum)
        file_groups.append(files_in_part)

    balance = (min(compile_per_partition) / max(compile_per_partition)
               if compile_per_partition and max(compile_per_partition) > 0 else 0.0)

    # 3. Compile time saving (sequential sum -> parallel max)
    before_ct = int(target_files['compile_time_ms'].sum() or 0)
    new_parallel_ct = max(compile_per_partition) if compile_per_partition else before_ct
    saving = before_ct - new_parallel_ct

    # 4. simulate_split for notes and cross_partition_edges
    sim = simulate_split(G, target_name, file_groups, target_df)

    return {
        'cmake_target': target_name,
        'n_partitions': len(partition_ids),
        'cross_partition_includes': cross_includes,
        'compile_per_partition': compile_per_partition,
        'balance_ratio': round(balance, 3),
        'before_compile_ms': before_ct,
        'new_parallel_ct_ms': new_parallel_ct,
        'compile_saving_ms': saving,
        'file_groups': file_groups,
        'notes': sim.get('notes', []),
    }


# ── Run evaluation for all candidates x 4 methods ───────────────────────────
eval_rows = []
for _, row in split_candidates.iterrows():
    t = row['cmake_target']
    if t not in G_contracted_map:
        continue
    n2s = supernode_map[t]
    gf = G_file_map[t]

    methods = {
        'spectral_2way': spectral_2way_map.get(t, {}),
        'spectral_kway': spectral_kway_map.get(t, (2, {}))[1],
        'metis_2way': metis_2way_map.get(t, {}),
        'metis_kway': metis_kway_map.get(t, (2, {}))[1],
    }

    for method_name, part in methods.items():
        if not part:
            continue
        ev = evaluate_partition(t, part, n2s, gf, file_df, target_df, G)
        ev['method'] = method_name
        # Merge constraint check
        cdf_match = constraint_df[
            (constraint_df['cmake_target'] == t)
            & (constraint_df['method'] == method_name)
        ] if not constraint_df.empty else pd.DataFrame()
        ev['codegen_ok'] = bool(cdf_match['ok'].all()) if not cdf_match.empty else True
        eval_rows.append(ev)

results_df = pd.DataFrame(eval_rows)

print(f"Evaluated {len(results_df)} (target, method) combinations")
display(results_df[['cmake_target', 'method', 'n_partitions', 'cross_partition_includes',
                     'balance_ratio', 'before_compile_ms', 'compile_saving_ms',
                     'codegen_ok']].reset_index(drop=True))

# ── Best partition selection per target ──────────────────────────────────────
# Score: higher is better. Balance * saving * (1 - cross_include penalty)
max_saving = results_df['compile_saving_ms'].max() if not results_df.empty else 1
max_cross = results_df['cross_partition_includes'].max() if not results_df.empty else 1
max_saving = max(max_saving, 1)
max_cross = max(max_cross, 1)

results_df['score'] = (
    results_df['balance_ratio']
    * (results_df['compile_saving_ms'] / max_saving)
    * (1 - results_df['cross_partition_includes'] / (max_cross + 1))
)

# Filter: balance >= 0.3, prefer codegen-ok
viable = results_df[results_df['balance_ratio'] >= 0.3].copy()
if viable.empty:
    viable = results_df.copy()

recommendations = {}
for t in viable['cmake_target'].unique():
    t_viable = viable[viable['cmake_target'] == t]
    # Prefer codegen-compliant partitions
    t_ok = t_viable[t_viable['codegen_ok']]
    if not t_ok.empty:
        best = t_ok.loc[t_ok['score'].idxmax()]
    else:
        best = t_viable.loc[t_viable['score'].idxmax()]
    recommendations[t] = best

print(f"\nRecommended partitions ({len(recommendations)} targets):")
for t, rec in recommendations.items():
    print(f"  {t}: {rec['method']} (k={rec['n_partitions']}, "
          f"balance={rec['balance_ratio']:.2f}, "
          f"saving={rec['compile_saving_ms']:,}ms, "
          f"cross_includes={rec['cross_partition_includes']}, "
          f"codegen_ok={rec['codegen_ok']})")

# ── Visualisation: 4-panel dashboard ─────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

targets_in_results = results_df['cmake_target'].unique()
method_palette = {'spectral_2way': 'steelblue', 'spectral_kway': '#4C9BE8',
                  'metis_2way': 'darkorange', 'metis_kway': '#E8A84C'}

# Panel 1: Cross-partition includes by method
ax = axes[0, 0]
if not results_df.empty:
    pivot_cross = results_df.pivot_table(index='cmake_target', columns='method',
                                         values='cross_partition_includes', aggfunc='first')
    pivot_cross = pivot_cross.reindex(columns=[m for m in method_names if m in pivot_cross.columns])
    x = np.arange(len(pivot_cross))
    w = 0.2
    for i, method in enumerate(pivot_cross.columns):
        ax.bar(x + i * w, pivot_cross[method].fillna(0), width=w,
               label=method, color=method_palette.get(method, 'grey'), alpha=0.8)
    ax.set_xticks(x + w * 1.5)
    ax.set_xticklabels([t[:25] for t in pivot_cross.index], rotation=45, ha='right', fontsize=7)
    ax.set_ylabel('Cross-partition includes')
    ax.set_title('Cross-Partition Include Count by Method')
    ax.legend(fontsize=7)

# Panel 2: Balance ratio by method
ax = axes[0, 1]
if not results_df.empty:
    pivot_bal = results_df.pivot_table(index='cmake_target', columns='method',
                                       values='balance_ratio', aggfunc='first')
    pivot_bal = pivot_bal.reindex(columns=[m for m in method_names if m in pivot_bal.columns])
    x = np.arange(len(pivot_bal))
    for i, method in enumerate(pivot_bal.columns):
        ax.bar(x + i * w, pivot_bal[method].fillna(0), width=w,
               label=method, color=method_palette.get(method, 'grey'), alpha=0.8)
    ax.axhline(0.5, color='red', linestyle='--', alpha=0.6, label='Target (0.5)')
    ax.set_xticks(x + w * 1.5)
    ax.set_xticklabels([t[:25] for t in pivot_bal.index], rotation=45, ha='right', fontsize=7)
    ax.set_ylabel('Balance ratio')
    ax.set_title('Compile Time Balance Ratio by Method')
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=7)

# Panel 3: Recommended partition compile time breakdown
ax = axes[1, 0]
if recommendations:
    rec_targets = list(recommendations.keys())
    cmap_parts = plt.cm.get_cmap('tab10')
    bottom = np.zeros(len(rec_targets))
    max_parts = max(len(recommendations[t]['compile_per_partition']) for t in rec_targets)
    for pid in range(max_parts):
        vals = []
        for t in rec_targets:
            cpp = recommendations[t]['compile_per_partition']
            vals.append(cpp[pid] if pid < len(cpp) else 0)
        ax.barh(range(len(rec_targets)), vals, left=bottom,
                color=cmap_parts(pid), label=f'Partition {pid}', alpha=0.8)
        bottom += np.array(vals)
    ax.set_yticks(range(len(rec_targets)))
    ax.set_yticklabels([t[:30] for t in rec_targets], fontsize=8)
    ax.set_xlabel('Compile time (ms)')
    ax.set_title('Recommended Split — Compile Time per Partition')
    ax.legend(fontsize=7, loc='lower right')

# Panel 4: Saving vs cross-partition includes
ax = axes[1, 1]
if not results_df.empty:
    for method in method_names:
        mdf = results_df[results_df['method'] == method]
        ax.scatter(mdf['compile_saving_ms'], mdf['cross_partition_includes'],
                   color=method_palette.get(method, 'grey'), label=method,
                   alpha=0.6, s=50)
    # Mark recommendations with stars
    for t, rec in recommendations.items():
        ax.scatter(rec['compile_saving_ms'], rec['cross_partition_includes'],
                   marker='*', s=200, color='red', zorder=5,
                   edgecolors='black', linewidths=0.5)
    ax.set_xlabel('Compile time saving (ms)')
    ax.set_ylabel('Cross-partition includes')
    ax.set_title('Saving vs New Dependencies (stars = recommended)')
    ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

# ── Export recommendations ───────────────────────────────────────────────────
export_rows = []
for t, rec in recommendations.items():
    export_rows.append({
        'cmake_target': t,
        'recommended_k': rec['n_partitions'],
        'method': rec['method'],
        'balance_ratio': rec['balance_ratio'],
        'cross_partition_includes': rec['cross_partition_includes'],
        'compile_saving_ms': rec['compile_saving_ms'],
        'before_compile_ms': rec['before_compile_ms'],
        'codegen_constraint_ok': rec['codegen_ok'],
        'file_groups_json': json.dumps(rec['file_groups']),
    })

if export_rows:
    export_df = pd.DataFrame(export_rows)
    Path('../data/results').mkdir(parents=True, exist_ok=True)
    export_df.to_parquet('../data/results/split_recommendations.parquet', index=False)
    print(f"\nSaved {len(export_df)} split recommendations to data/results/split_recommendations.parquet")
    display(export_df[['cmake_target', 'recommended_k', 'method', 'balance_ratio',
                        'cross_partition_includes', 'compile_saving_ms',
                        'codegen_constraint_ok']].reset_index(drop=True))
else:
    print("\nNo viable split recommendations to export.")